# Kubeflow Pipeline

Develop the YOLO training pipeline, one step at a time.

```txt
fetch_data -> prepare_data -> train -> evaluate -> upload_model
```


## Environment

In [ ]:
# pip install
%pip install -q -U kfp

In [ ]:
from kfp import dsl
from kfp.dsl import Dataset, Input, Output


---

## Step 1: `fetch_data`

Pull the DVC-tracked raw dataset out of S3.


In [ ]:
@dsl.component(base_image="python:3.12", packages_to_install=["boto3"])
def fetch_data(
    bucket: str,
    dvc_dir_hash: str,
    region: str,
    raw: Output[Dataset],
):
    """Restore the DVC-tracked raw dataset from S3 into the `raw` artifact."""
    import json
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path

    import boto3

    s3 = boto3.client("s3", region_name=region)

    def dvc_key(md5: str) -> str:
        # DVC shards its content-addressed store by the first two hex chars.
        return "dvcstore/files/md5/" + md5[:2] + "/" + md5[2:]

    # the .dir object lists {"md5": ..., "relpath": ...} for every file
    manifest = json.loads(
        s3.get_object(Bucket=bucket, Key=dvc_key(dvc_dir_hash))["Body"].read()
    )

    root = Path(raw.path)
    root.mkdir(parents=True, exist_ok=True)

    def fetch(entry):
        target = root / entry["relpath"]
        target.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, dvc_key(entry["md5"]), str(target))

    # 1100+ small objects: this is latency-bound, not bandwidth-bound
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(fetch, manifest))

    images = [p for p in root.iterdir() if p.suffix.lower() in {".jpeg", ".jpg", ".png"}]
    if not images:
        raise RuntimeError("no images restored -- check the dvc_dir_hash")

    # metadata shows up on the artifact in the run UI
    raw.metadata["files"] = len(manifest)
    raw.metadata["images"] = len(images)
    raw.metadata["dvc_dir_hash"] = dvc_dir_hash

    print("fetched", len(manifest), "files /", len(images), "images to", root)

---

## Step 2: `prepare_data`

Split into train/val and write the ultralytics descriptor.


In [ ]:
@dsl.component(base_image="python:3.12", packages_to_install=["pyyaml"])
def prepare_data(
    raw: Input[Dataset],
    val_fraction: float,
    split_seed: int,
    processed: Output[Dataset],
):
    """Build processed/{train,val}/{images,labels} plus data.yaml."""
    import random
    import shutil
    from pathlib import Path

    import yaml

    suffixes = {".jpeg", ".jpg", ".png"}
    src = Path(raw.path)
    dst = Path(processed.path)

    # images and labels pair by basename: foo.jpeg <-> foo.txt
    stems = sorted(p.stem for p in src.iterdir() if p.suffix.lower() in suffixes)
    random.Random(split_seed).shuffle(stems)
    cut = int(len(stems) * (1 - val_fraction))

    counts = {}
    for split, names in (("train", stems[:cut]), ("val", stems[cut:])):
        for sub in ("images", "labels"):
            (dst / split / sub).mkdir(parents=True, exist_ok=True)
        for stem in names:
            image = next(p for p in src.glob(stem + ".*") if p.suffix.lower() in suffixes)
            shutil.copy(image, dst / split / "images" / image.name)
            label = src / (stem + ".txt")
            # an image with no label file is a legitimate negative sample
            if label.exists():
                shutil.copy(label, dst / split / "labels" / label.name)
        counts[split] = len(names)

    if not counts["train"] or not counts["val"]:
        raise RuntimeError("empty split: " + str(counts))

    class_names = (src / "classes.txt").read_text().split()
    (dst / "data.yaml").write_text(
        yaml.safe_dump(
            {
                "path": str(dst),          # absolute, see above
                "train": "train/images",
                "val": "val/images",
                "nc": len(class_names),
                "names": class_names,
            },
            sort_keys=False,
        )
    )

    processed.metadata.update(counts)
    processed.metadata["classes"] = class_names
    print("split", counts, "classes", class_names)
    print((dst / "data.yaml").read_text())

---

## Next

Wire the two steps into a `@dsl.pipeline` and compile.
